In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
import json
import copy
from matplotlib.image import imread
def load_spatial(path, adata, library_id='0'):
    tissue_positions_file = join(path, "tissue_positions.csv")
    files = dict(
        tissue_positions_file=tissue_positions_file,
        scalefactors_json_file=join(path, "scalefactors_json.json"),
        hires_image=join(path, "tissue_hires_image.png"),
        lowres_image=join(path, "tissue_lowres_image.png"),
    )
    
    adata.uns["spatial"] = dict()
    adata.uns["spatial"][library_id] = dict()
    adata.uns["spatial"][library_id]["images"] = dict()
    for res in ["hires", "lowres"]:
        try:
            adata.uns["spatial"][library_id]["images"][res] = imread(
                str(files[f"{res}_image"])
            )
        except Exception:
            raise OSError(f"Could not find '{res}_image'")

    # read json scalefactors
    adata.uns["spatial"][library_id]["scalefactors"] = json.loads(
        Path(files["scalefactors_json_file"]).read_bytes()
    )

    # adata.uns["spatial"][library_id]["metadata"] = {
    #     k: (str(attrs[k], "utf-8") if isinstance(attrs[k], bytes) else attrs[k])
    #     for k in ("chemistry_description", "software_version")
    #     if k in attrs
    # }

    # read coordinates
    positions = pd.read_csv(
        files["tissue_positions_file"],
        header=0 if Path(tissue_positions_file).name == "tissue_positions.csv" else None,
        index_col=0,
    )
    positions.columns = [
        "in_tissue",
        "array_row",
        "array_col",
        "pxl_col_in_fullres",
        "pxl_row_in_fullres",
    ]
    # print(positions.head())

    adata.obs = adata.obs.join(positions, how="left")

    adata.obsm["spatial"] = adata.obs[
        ["pxl_row_in_fullres", "pxl_col_in_fullres"]
    ].to_numpy()
   
    adata.obs.drop(
        columns=["pxl_row_in_fullres", "pxl_col_in_fullres"],
        inplace=True,
    )

import gzip
from scipy.io import mmread
from pathlib import Path, PurePath
def load_data(_dir):
    feat_names = pd.read_csv(join(_dir, 'features.tsv.gz'), compression='gzip', sep='\t', header=None)
    barcodes   = pd.read_csv(join(_dir, 'barcodes.tsv.gz'), compression='gzip', sep='\t', header=None)

    with gzip.open(join(_dir, 'matrix.mtx.gz'), 'rb') as gzipped_file:
        mat = mmread(gzipped_file)

    ad = sc.AnnData(sps.csr_matrix(mat.T))
    ad.obs_names = barcodes[0].values
    ad.var_names = feat_names[1].values
    ad.var['id'] = feat_names[0].values
    ad.var['type'] = feat_names[2].values
    return ad

In [3]:
data_dir = '../../../data/raw/Lymph_node/LN-2024-new/outs'

ad3 = load_data(join(data_dir, 'filtered_feature_bc_matrix'))
ad3_rna = ad3[:, ad3.var['type']=='Gene Expression'].copy()
ad3_adt = ad3[:, ad3.var['type']=='Antibody Capture'].copy()
load_spatial(join(data_dir, 'spatial'), ad3_rna)
load_spatial(join(data_dir, 'spatial'), ad3_adt)

ad3_rna.obs['src'] = ad3_adt.obs['src'] = ['s3']*ad3_rna.n_obs
ad3_rna.obs_names = [f's3-{x}' for x in ad3_rna.obs_names]
ad3_adt.obs_names = [f's3-{x}' for x in ad3_adt.obs_names]

ad3_rna.var_names_make_unique()
ad3_adt.var_names_make_unique()

data_dir = '../../../data/raw/Lymph_node/lymp_tonsil_ramen'

ad_a1_rna = sc.read_h5ad(join(data_dir, 'lymph_A1/adata_RNA.h5ad'))
ad_a1_adt = sc.read_h5ad(join(data_dir, 'lymph_A1/adata_ADT.h5ad'))
meta1 = pd.read_csv(join(data_dir, 'lymph_A1/A1_LN_cloupe_Kwoh.csv'), index_col=0) 
ad_a1_rna.obs['lab'] = meta1.loc[ad_a1_rna.obs_names, 'manual'].to_list()
ad_a1_adt.obs['lab'] = meta1.loc[ad_a1_adt.obs_names, 'manual'].to_list()
ad_a1_rna.obs['src'] = ad_a1_adt.obs['src'] = ['s1'] * ad_a1_rna.n_obs
ad_a1_rna.obs_names = [f's1-{x}' for x in ad_a1_rna.obs_names]
ad_a1_adt.obs_names = [f's1-{x}' for x in ad_a1_adt.obs_names]
ad_a1_rna.var_names_make_unique()
ad_a1_adt.var_names_make_unique()

ad_d1_rna = sc.read_h5ad(join(data_dir, 'lymph_D1/adata_RNA.h5ad'))
ad_d1_adt = sc.read_h5ad(join(data_dir, 'lymph_D1/adata_ADT.h5ad'))
meta2 = pd.read_csv(join(data_dir, 'lymph_D1/D1_LN_cloupe_Kwoh.csv'), index_col=0) 
ad_d1_rna.obs['lab'] = meta2.loc[ad_d1_rna.obs_names, 'manual'].to_list()
ad_d1_adt.obs['lab'] = meta2.loc[ad_d1_adt.obs_names, 'manual'].to_list()
ad_d1_rna.obs['src'] = ad_d1_adt.obs['src'] = ['s2'] * ad_d1_rna.n_obs
ad_d1_rna.obs_names = [f's2-{x}' for x in ad_d1_rna.obs_names]
ad_d1_adt.obs_names = [f's2-{x}' for x in ad_d1_adt.obs_names]
ad_d1_rna.var_names_make_unique()
ad_d1_adt.var_names_make_unique()

## unify feature names
shared_gene = ad_a1_rna.var_names.intersection(ad_d1_rna.var_names).intersection(ad3_rna.var_names)
shared_prot = ad_a1_adt.var_names.intersection(ad_d1_adt.var_names).intersection(ad3_adt.var_names)

ad_a1_rna, ad_d1_rna, ad3_rna = ad_a1_rna[:, shared_gene].copy(), ad_d1_rna[:, shared_gene].copy(), ad3_rna[:, shared_gene].copy()
ad_a1_adt, ad_d1_adt, ad3_adt = ad_a1_adt[:, shared_prot].copy(), ad_d1_adt[:, shared_prot].copy(), ad3_adt[:, shared_prot].copy()

/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages

In [4]:
ad_rna_all = sc.concat([ad_a1_rna, ad_d1_rna, ad3_rna])
ad_adt_all = sc.concat([ad_a1_adt, ad_d1_adt, ad3_adt])

sc.pp.highly_variable_genes(ad_rna_all, batch_key="src", flavor="seurat_v3", n_top_genes=5000)

ad_a1_rna = ad_a1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad_d1_rna = ad_d1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad3_rna = ad3_rna[:, ad_rna_all.var.query('highly_variable').index].copy()

In [5]:
RNA_ADS = [ad_a1_rna, ad_d1_rna, ad3_rna]
ADT_ADS = [ad_a1_adt, ad_d1_adt, ad3_adt]
n_batches = 3
mod_sets = ['rna', 'adt']
mod_dict = {'rna': RNA_ADS, 'adt': ADT_ADS}

In [6]:
res = []
for i in range(n_batches):  # train test split
    print(f'cv={i+1}, missing=adt')
    mod_BatchDict = {'rna': RNA_ADS, 
                     'adt': [ADT_ADS[bi] if bi!=i else None for bi in range(n_batches)]}
    input_key = 'dimred_bc'
    batch_key = 'src'
        
    ADT_preprocess(mod_BatchDict['adt'], batch_corr=True, batch_key=batch_key, key=input_key)
    RNA_preprocess(mod_BatchDict['rna'], favor='scanpy', batch_corr=True, n_hvg=5000, batch_key=batch_key, key=input_key)
    
    model = SpaMosaic(
        modBatch_dict=mod_BatchDict, input_key=input_key,
        batch_key=batch_key, 
        intra_knns=10, inter_knn_base=10, 
        w_g=0.8,
        seed=1234, 
        device='cuda:0'
    )
    
    model.train(net='wlgcn', lr=0.01, T=0.01, n_epochs=100)
    _ = model.infer_emb(mod_BatchDict, emb_key='emb', final_latent_key='merged_emb')

    for k in mod_BatchDict.keys():
        for ad in mod_BatchDict[k]:
            if ad is not None:
                ad.layers['counts'] = sps.csr_matrix(ad.X)

    # for knn in [10, 20, 30]:
    imp_dict = model.impute(mod_BatchDict, emb_key='emb', layer_key='counts', imp_knn=10)

    print(f'==> cv {i}, ADT imputation: ')
    pr_X = imp_dict['adt'][i] 
    ad_pred = sc.AnnData(pr_X, obs=mod_dict['adt'][i].obs.copy(), var=mod_dict['adt'][i].var.copy())

    out_dir = "../../../results/imputations/SpaMosaic/Lymph_imputation/" #path to results
    os.makedirs(out_dir, exist_ok=True)
    ad_pred.write_h5ad(f'{out_dir}/cv{i}_imputedADT.h5ad')

cv=1, missing=adt
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
batch0: ['rna']
batch1: ['rna', 'adt']
batch2: ['rna', 'adt']
------Calculating spatial graph...
The graph contains 34840 edges, 3484 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 33590 edges, 3359 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34080 edges, 3408 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 33590 edges, 3359 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34080 edges, 3408 cells.
10.0000 n

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 26.84it/s]


impute adt-counts for batch-1
==> cv 0, ADT imputation: 
cv=2, missing=adt
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['rna', 'adt']
------Calculating spatial graph...
The graph contains 34840 edges, 3484 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 33590 edges, 3359 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34080 edges, 3408 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34840 edges, 3484 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:04<00:00, 22.43it/s]


impute adt-counts for batch-2
==> cv 1, ADT imputation: 
cv=3, missing=adt
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
Reach convergence after 3 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna', 'adt']
batch2: ['rna']
------Calculating spatial graph...
The graph contains 34840 edges, 3484 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 33590 edges, 3359 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34080 edges, 3408 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 34840 edges, 3484 cells.
10.0000 neighbors per cell on average.

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 26.28it/s]


impute adt-counts for batch-3
==> cv 2, ADT imputation: 
